In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# RMI Tutorial 04: Pub/Sub Real-time Verification

RMI provides near real-time traffic updates via Cloud Pub/Sub. This tutorial uses the `gcloud` CLI to verify your real-time data stream.

### Objectives:
1. **Identify** your RMI provider topic.
2. **Create** a local Pull subscription.
3. **Retrieve** and parse real-time messages via the CLI.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/tutorials/04_pubsub_verification.ipynb) 
[![Open In Colab Enterprise](https://img.shields.io/badge/Open%20in-Colab%20Enterprise-blue?logo=google-cloud&logoColor=white)](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Ftutorials%2F04_pubsub_verification.ipynb) 
[![Open In BigQuery Notebooks](https://img.shields.io/badge/Open%20in-BigQuery%20Notebooks-blue?logo=google-cloud&logoColor=white)](https://console.cloud.google.com/bigquery/import?url=https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Ftutorials%2F04_pubsub_verification.ipynb) 
[![View on GitHub](https://img.shields.io/badge/View%20on-GitHub-lightgrey?logo=github&logoColor=white)](https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/tutorials/04_pubsub_verification.ipynb)


In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

## 1. Environment Setup
Set your Project ID and Project Number (from Tutorial 01).

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID" # @param {type:"string"}
PROJECT_NUMBER = "YOUR_PROJECT_NUMBER" # @param {type:"string"}
SUBSCRIPTION_ID = "tutorial-rmi-json-pull-sub"

import os
os.environ["PROJECT_ID"] = PROJECT_ID
# Note: Topic names are provided by the RMI team and may vary.
os.environ["TOPIC_PATH"] = f"projects/maps-platform-roads-management/topics/roadsinformation-{PROJECT_NUMBER}-json"
os.environ["SUBSCRIPTION_PATH"] = f"projects/{PROJECT_ID}/subscriptions/{SUBSCRIPTION_ID}"

## 2. Create the Pull Subscription
We create a subscription in your local project to listen to the RMI provider topic.

In [ ]:
%%bash
gcloud pubsub subscriptions create "${SUBSCRIPTION_PATH}" \
  --topic="${TOPIC_PATH}" \
  --project="${PROJECT_ID}"

## 3. Pull and Parse Messages
Retrieve messages from the stream. We use `--auto-ack` to automatically acknowledge receipt and `--format="json"` to view the full RMI payload.

In [ ]:
%%bash
gcloud pubsub subscriptions pull "${SUBSCRIPTION_PATH}" \
  --limit=5 \
  --auto-ack \
  --format="json" \
  --project="${PROJECT_ID}"

## 4. Optional: Create a BigQuery Subscription
For long-term storage and analysis, you can stream RMI updates directly into a BigQuery table.

### When to use a BigQuery Subscription vs. RMI Shared Tables?
RMI already provides a **`recent_roads_data`** table via your Analytics Hub subscription. 

**Use the RMI Shared Table (`recent_roads_data`) if:**
- You only need the last 24 hours of data.
- You want a zero-maintenance, managed experience.

**Use a BigQuery Subscription (Custom) if:**
- You need to **retain** data for more than 24 hours in a raw format.
- You want to perform **custom real-time transformations** during ingestion.

### Create the Target Dataset

In [ ]:
%%bash
bq mk --project_id=${PROJECT_ID} --dataset rmi_realtime_stream

### Grant Permissions to Pub/Sub Service Account
Pub/Sub requires the `BigQuery Data Editor` role on your project (or dataset) to write messages into the table. The service account follows the pattern: `service-PROJECT_NUMBER@gcp-sa-pubsub.iam.gserviceaccount.com`.

In [ ]:
%%bash
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:service-${PROJECT_NUMBER}@gcp-sa-pubsub.iam.gserviceaccount.com" \
    --role="roles/bigquery.dataEditor"

### Create the BigQuery Subscription
This creates a no-code streaming pipeline from Pub/Sub to BigQuery.

In [ ]:
%%bash
gcloud pubsub subscriptions create "projects/${PROJECT_ID}/subscriptions/tutorial-rmi-bq-sub" \
  --topic="${TOPIC_PATH}" \
  --bigquery-table="${PROJECT_ID}:rmi_realtime_stream.realtime_updates" \
  --use-topic-schema \
  --project="${PROJECT_ID}"

## 5. Optional: Subscription Filtering
You can use [Subscription Filters](https://docs.cloud.google.com/pubsub/docs/subscription-message-filter) to only receive a subset of messages.

### Create a Filtered Subscription
Only receives messages for a specific route ID.

In [ ]:
%%bash
# Create a subscription with prefix filtering
# This only receives messages where the route ID starts with "PREFIX-"
gcloud pubsub subscriptions create "projects/${PROJECT_ID}/subscriptions/tutorial-rmi-prefix-filtered-sub" \
  --topic="${TOPIC_PATH}" \
  --message-filter='attributes.selected_route_id.hasPrefix("PREFIX-")' \
  --project="${PROJECT_ID}"


## Summary & Next Steps
You have successfully verified your real-time data stream using the `gcloud` CLI.

**Next**: In the next tutorial, we will **Visualize** our RMI data on an interactive map using **pydeck**.

---
**For more advanced RMI query patterns, visit the official [RMI Sample Queries Repository](https://github.com/googlemaps-samples/insights-samples/tree/main/roads_management_insights/rmi_sample_queries).**